In [2]:
%pwd

'c:\\Users\\DELL\\Desktop\\ChatBot\\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws\\analysis'

In [3]:
import os
os.chdir("../")
%pwd

'c:\\Users\\DELL\\Desktop\\ChatBot\\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader ,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [ ]:

def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [ ]:
extracted_data = load_pdf_file("data/test_data")


In [53]:
from typing import  List 
from langchain.schema import Document

def filter_to_minimal_docs(docs:List[Document]) -> List[Document]:
    minimal_docs : List[Document] =[]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(Document(page_content =doc.page_content,
                            metadata={"source":src})
        )
    return minimal_docs
        




In [54]:
minimal_data = filter_to_minimal_docs(extracted_data)

266

In [84]:
#create Chunks from data
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap =20        
    )
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

In [85]:
texts_chunk = text_split(minimal_data)
print(len(texts_chunk))

579


In [88]:
#embedding 
model_name = "sentence-transformers/all-MiniLM-L6-v2"
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
        
    )
    return embeddings
embeddings = download_embeddings()

c:\Users\DELL\Desktop\ChatBot\Chat_bot_LLm_Pinecone_Langchain_Flask_Aws\chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [90]:
import os
from  dotenv import load_dotenv
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPEN_API_KEY = os.getenv("OPEN_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPEN_API_KEY"] =  OPEN_API_KEY


In [91]:
from  pinecone import Pinecone
pinecone_api_key =PINECONE_API_KEY
pc = Pinecone(api_key= pinecone_api_key)

In [ ]:
from pinecone import ServerlessSpec
index_name  ="chatbot"
# Delete the existing index 

if pc.has_index(index_name):
    pc.delete_index(index_name)
    print(f"Deleted existing index: {index_name}")

if not pc.has_index(index_name):
    pc.create_index(name = index_name
                 ,dimension=384 
                ,metric = "cosine" 
                ,spec = ServerlessSpec(cloud ="aws",
                                       region ="us-east-1")
                 )


#####  Create New index 
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents = texts_chunk
    ,embedding = embeddings
    ,index_name = index_name    
)
print(f"Created new index: {index_name} with dimension 384")

######  Load existing index
#docsearch = PineconeVectorStore.from_existing_index()
#docsearch.add_documents(texts_chunk)

Created new index: chatbot with dimension 384


In [93]:
# Load Existing index 
index_name  ="chatbot"
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [94]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})


In [96]:
retrieved_docs = retriever.invoke("What is largest mountains range in indai?")
retrieved_docs

[Document(page_content='14 \n \nGreater Himalayas  \n\uf0b7 These are also known as Inner Himalayas or Himadri. This is the highest range of \nHimalayas. The average height of this range is 6,100 m.  It contains all the major ranges \nof the Himalayas. It ranges 120 km to 190 km.  \n\uf0b7 Great Himalayan range, also known as the Central Axial range, extends from the gorge of \nInd us River to the bend of Brahmaputra river in Arunachal Pradesh.  \n\uf0b7 Almost all the lofty peaks of the world are located in this range. Mt Everest, \nKanchenjunga, Nanga Parbat, Nanda Devi, Kamet and Namcha Barwa are its important \nrange.  \n \nMiddle or the lesser H imalayas  \n\uf0b7 These are also known as Himachal.  \n\uf0b7 Greater Himalayas is separated from the Middle Himalayas by the Main Central Thrust. \nIts breadth is 60 - 80 km and average height is 3,000 - 4,500m.  \n\uf0b7 Some peaks in this range are more than 5,000 m high and the river f low through deep \ngorges upto 1,000 m.', metadat

In [97]:
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

chatModel = ChatOpenAI(
    model="gpt-4o",
    api_key=OPEN_API_KEY
)

In [98]:
system_prompt = (
    "You are an Geograpy assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [99]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [100]:
response = rag_chain.invoke({"input": "what is mountains?"})
print(response["answer"])

Mountains are natural landforms that rise prominently above their surroundings, typically featuring steep slopes and a significant elevation difference compared to adjacent terrain. They are formed through tectonic forces, volcanic activity, or erosion and often have a peak or summit. Mountains can be found worldwide and are often associated with diverse ecosystems and climates due to their varying altitudes.


In [101]:
response = rag_chain.invoke({"input": "where is galwan valley situated"})
print(response["answer"])

Galwan Valley is situated in the Union Territory of Ladakh, India. It is located near the Line of Actual Control (LAC) between India and China. The valley has been a point of strategic importance and contention between the two countries.


In [102]:
response = rag_chain.invoke({"input": "what is operation sindoor by indian army"})
print(response["answer"])

I don't know about Operation Sindoor by the Indian Army.
